# Chapter 10: The locked door

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "10"
__file__ = os.path.abspath("run.py")

## The scene

Stairwell C has two doors. At 23:30 every night the bottom one locks itself. At 23:36 someone opened
it from the quay side. The badge log does not show a name, only a code. Varma has the facilities
manager on the phone and the site rules on her desk. "Before I ask anyone a question in an interview
room," she says, "I want the list of people who could have opened that door. And the list of people
who could not. With reasons."

The evidence: the door rules in [`case/site-facts.md`](../../case/site-facts.md), the
[`case/badge-log.csv`](../../case/badge-log.csv) and the hut camera rows in
[`case/gate-camera-log.csv`](../../case/gate-camera-log.csv).

## The concept

In chapter 09 you asked for reasoning in the answer. A **thinking model** is trained to do that on
its own, in a separate channel, before it writes the answer. Qwen3, which LANTERN runs, is a
**hybrid** model: the same weights can answer directly (`think=False`) or think first
(`think=True`). Ollama returns the two parts separately, so `reply.thinking` holds the reasoning
and `reply.text` the answer.

```mermaid
flowchart LR
    P[Prompt] --> M{think?}
    M -- False --> A1[Answer tokens]
    M -- True --> T["Thinking tokens<br/>(reasoning channel)"] --> A2[Answer tokens]
    T -.->|returned as<br/>reply.thinking| U[You]
    A1 & A2 -.->|reply.text| U
```

Frontier APIs have the same idea under other names: extended thinking with a token budget,
reasoning effort settings, "thinking" model variants. The trade is always the same: more tokens and
more time for a better chance at multi-step problems.

## How it works

Thinking tokens are output tokens. They come out of the same `max_tokens` (Ollama: `num_predict`)
budget as the answer, they cost the same time to generate, and on a paid API they are billed. If the
budget runs out while the model is still thinking, you get an empty answer.

![Thinking budget](../../docs/diagrams/10-thinking-budget.svg)

The script asks the same access puzzle three times on `qwen3:1.7b`:

1. `think=False`, `max_tokens=700`.
2. `think=True`, `max_tokens=300`, a budget too small to finish thinking.
3. `think=True`, `max_tokens=4000`, temperature 0.6 (Qwen's recommended setting for thinking).

With `--large`, step 3 runs on `qwen3:4b` instead.

```mermaid
sequenceDiagram
    participant You
    participant Ollama
    You->>Ollama: chat(puzzle, think=True, max_tokens=300)
    Ollama-->>You: thinking: "Okay, let's tackle these questions step by step..." (300 tokens)
    Ollama-->>You: text: "" (budget gone)
    You->>Ollama: chat(puzzle, think=True, max_tokens=4000)
    Ollama-->>You: thinking: about 6,300 characters
    Ollama-->>You: text: the answer
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import csv
import io
import re
import sys

from lantern import case, chat, learn
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

chapter("10", "The locked door", """
Stairwell C has two doors. At 23:30 every night the bottom one locks itself. At 23:36 someone
opened it from the quay side. The badge log does not show a name, only a code. Varma has the
facilities manager on the phone and the site rules on her desk. "Before I ask anyone a question
in an interview room," she says, "I want the list of people who could have opened that door. And
the list of people who could not. With reasons."
""")

LARGE = "--large" in sys.argv  # run the thinking step on qwen3:4b (slow on a CPU)

facts = case.read("site-facts.md")
rules = "\n".join(line for line in facts.splitlines() if line.startswith(
    ("- From 23:30", "  without", "  from the", "- Level 3", "- Override", "  OVR-")))
badge = "\n".join(line for line in case.read("badge-log.csv").splitlines()
                  if line.startswith(("time", "23:2", "23:3")))
hut = "\n".join(line for line in case.read("gate-camera-log.csv").splitlines()
                if "HUT" in line or line.startswith("time"))
evidence("site-facts.md (door rules)", rules)
evidence("badge-log.csv (23:28 to 23:37)", badge)

puzzle = f"""<door_rules>
{rules}
</door_rules>
<badge_log>
{badge}
</badge_log>
<hut_camera>
{hut}
</hut_camera>
<other_facts>
- E. Voss was at the Harbour Hotel dinner from 19:30 until after midnight, seen by hundreds.
- T. Reyes badged into STAIR-C-TOP at 23:13 and was inside the stairwell.
- The figure in hi-vis on the hut camera is I. Petrov.
</other_facts>

At 23:36:15 the bottom door of stairwell C was opened from the outside.
Q1. Which credential opened it?
Q2. For each candidate, could they have been the one who opened it from the outside at 23:36:15?
Write COULD or COULD NOT and the record that decides it. Use where each person was and which
credential was used.
  a) T. Reyes
  b) R. Malhotra
  c) E. Voss
  d) I. Petrov
  e) the lift maintenance contractor (holder of OVR-MNT-01)
  f) Brightwater & Co (holder of OVR-AUD-02)
Q3. One sentence: who must be interviewed about this door first?"""

runs = []


def run(label: str, think: bool, max_tokens: int, model: str | None = None) -> None:
    r = chat(puzzle, model=model, think=think, max_tokens=max_tokens,
             temperature=0.6 if think else 0)
    runs.append((label, r))
    if r.thinking:
        excerpt = r.thinking[:1400] + ("\n\n[...]" if len(r.thinking) > 1400 else "")
        reply(f"{label}: thinking (first 1400 chars of {len(r.thinking)})", excerpt,
              style="magenta")
    reply(f"{label}: answer", r.text or "(empty: the budget ran out while it was still thinking)",
          meta=f"{r.output_tokens} tokens out, {r.seconds:.0f}s",
          style="green" if r.text else "red")


note("The prompt is the `puzzle` variable in run.py. Every run below sends it.")

### 1. Thinking off

In [ ]:
step("1. Thinking off")
note("think=False: the model starts writing the answer at once.")
run("think off", think=False, max_tokens=700)

### 2. Thinking on, small budget

In [ ]:
step("2. Thinking on, small budget")
note("think=True with max_tokens=300. Thinking tokens come out of the same budget as the answer.")
p_small = learn.predict(
    "Thinking on, and 300 tokens in all for thinking and answer. What will come back?",
    ["a complete answer", "an answer cut off part way",
     "no answer at all: the budget runs out while it is still thinking"])
run("think on, 300 tokens", think=True, max_tokens=300)
small = runs[-1][1]
learn.reveal(
    p_small,
    answer=2 if not small.text else (1 if small.stop_reason in ("length", "max_tokens") else 0),
    why=f"It wrote {len(small.thinking):,} characters of thinking and {len(small.text):,} of "
    "answer. Thinking tokens are output tokens: they are spent first, from the same max_tokens.",
)

### 3. Thinking on, room to finish

In [ ]:
step("3. Thinking on, room to finish")
note("think=True with max_tokens=4000. Qwen recommends temperature 0.6 for thinking mode.")
if LARGE:
    note("--large: this step runs on qwen3:4b. Expect several minutes on a CPU.")
p_long = learn.predict_number(
    "With 4,000 tokens of room, how many output tokens (thinking and answer together) will it "
    "use?", unit="tokens")
run("think on, 4000 tokens" + (" (4b)" if LARGE else ""), think=True, max_tokens=4000,
    model="large" if LARGE else None)
learn.reveal(p_long, actual=runs[-1][1].output_tokens, tolerance=0.3,
             why="A thinking model decides for itself how long to think, and on a puzzle with six "
             "candidates it thinks at length. Budget for the long tail, not the average.")

### 4. What thinking cost

In [ ]:
step("4. What thinking cost")
table = Table(title="Same puzzle, three runs")
for col in ("run", "thinking chars", "answer chars", "tokens out", "seconds"):
    table.add_column(col, justify="left" if col == "run" else "right")
for label, r in runs:
    table.add_row(label, str(len(r.thinking)), str(len(r.text)), str(r.output_tokens),
                  f"{r.seconds:.0f}")
console.print(table)
note("Check every COULD and COULD NOT against the records. "
     "Thinking makes a model slower, not honest.")

# Step 5: the same question answered by code, from the same records, to score every run.
OVERRIDES = {
    "OVR-MNT-01": {"holder": "the lift maintenance contractor", "valid": ("08:00", "18:00")},
    "OVR-AUD-02": {"holder": "Brightwater & Co", "valid": None},  # never revoked
}
ALIBIS = {
    "T. Reyes": "inside the stairwell from 23:13 (STAIR-C-TOP); the door was opened from outside",
    "R. Malhotra": None,  # badged out of admin at 23:31, nothing places him at 23:36
    "E. Voss": "at the Harbour Hotel dinner from 19:30, seen by hundreds",
    "I. Petrov": "on the hut camera from 23:29:50 to 23:44:12",
}


def hhmm(t: str) -> int:
    h, m = (int(x) for x in t.split(":")[:2])
    return h * 60 + m


def door_candidates(log_csv: str, *, door: str, at: str, overrides: dict,
                    alibis: dict) -> dict:
    """Who could have opened `door` from outside at `at`. The exercise in practice/ builds this."""
    row = next((r for r in csv.DictReader(io.StringIO(log_csv))
                if r["door"] == door and r["time"] == at and r["direction"] == "in"), None)
    if row is None:
        raise ValueError(f"no entry through {door} at {at} in the log")
    cred = row["badge_id"]
    could, ruled_out = [], {}
    for code, o in overrides.items():
        start, end = o["valid"] or ("00:00", "23:59")
        valid = hhmm(start) <= hhmm(at) <= hhmm(end)
        hours = "" if o["valid"] is None else f"valid {start} to {end} only"
        if code == cred and valid:
            could.append(o["holder"])
        elif code == cred:
            ruled_out[o["holder"]] = f"{code} is {hours}"
        else:
            ruled_out[o["holder"]] = f"{code} was not used" + (f", and is {hours}" if hours else "")
    if cred not in overrides:
        could.append(row["holder"])
    for person, alibi in alibis.items():
        if person not in could and alibi:
            ruled_out[person] = alibi
    not_ruled_out = [p for p, a in alibis.items() if not a and p not in could]
    return {"credential": cred, "could": could, "ruled_out": ruled_out,
            "not_ruled_out": not_ruled_out}


LETTER = {"a": "T. Reyes", "b": "R. Malhotra", "c": "E. Voss", "d": "I. Petrov",
          "e": "the lift maintenance contractor", "f": "Brightwater & Co"}


def verdicts(text: str) -> dict[str, str]:
    """COULD or COULD NOT for each lettered candidate a) to f) in a model's answer."""
    parts = re.split(r"(?im)^[\W_]*([a-f])\)", text)
    out = {}
    for letter, body in zip(parts[1::2], parts[2::2], strict=False):
        # The first verdict word decides: a closing summary ("All candidates COULD NOT except
        # f)") lands in the last candidate's body and must not overrule its own verdict.
        m = re.search(r"COULD\s+NOT|COULDN'T|CANNOT|COULD", body.upper())
        if m:
            out[letter.lower()] = "COULD" if m.group(0) == "COULD" else "COULD NOT"
    return out

### 5. The same question, answered by code

In [ ]:
step("5. The same question, answered by code")
p_best = learn.predict(
    "Code now answers Q2 from the records: who could, who could not. Of the six candidates, how "
    "many will the best of the three LANTERN answers get right?",
    ["0 to 2", "3 or 4", "5", "all 6"])
truth = door_candidates(case.read("badge-log.csv"), door="STAIR-C-BOTTOM", at="23:36:15",
                        overrides=OVERRIDES, alibis=ALIBIS)
expected = {k: "COULD" if v in truth["could"] or v in truth["not_ruled_out"] else "COULD NOT"
            for k, v in LETTER.items()}
check = Table(title=f"Opened with {truth['credential']}: who could have been at the door")
for col in ("candidate", "by the records", "reason", *[label for label, _ in runs]):
    check.add_column(col)
scores = []
for _, r in runs:
    got = verdicts(r.text)
    scores.append(sum(got.get(k) == v for k, v in expected.items()))
for k, person in LETTER.items():
    reason = truth["ruled_out"].get(person) or (
        "holds the code that opened it" if person in truth["could"]
        else "nothing places them elsewhere; could have had the code")
    check.add_row(f"{k}) {person}", expected[k], reason,
                  *[verdicts(r.text).get(k, "-") for _, r in runs])
check.add_row("right of 6", "", "", *[str(n) for n in scores])
console.print(check)
learn.reveal(p_best, answer=learn.bucket(max(scores), [2, 4, 5]),
             why="The records decide five of the six candidates outright and leave one open. A "
             "dozen lines of code get every one right in milliseconds; use the check to score the "
             "model, not the model to replace the check.")
credential_right = [truth["credential"] in r.text for _, r in runs]
learn.record(
    think_off_right=scores[0],
    small_budget_right=scores[1],
    long_budget_right=scores[2],
    best_right=max(scores),
    credential_named=sum(credential_right),
    think_off_tokens=runs[0][1].output_tokens,
    small_budget_tokens=runs[1][1].output_tokens,
    small_budget_answer_chars=len(runs[1][1].text),
    long_budget_tokens=runs[2][1].output_tokens,
    long_budget_answer_chars=len(runs[2][1].text),
    long_budget_thinking_chars=len(runs[2][1].thinking),
    think_off_seconds=round(runs[0][1].seconds, 1),
    long_budget_seconds=round(runs[2][1].seconds, 1),
    large=LARGE,
)

your_turn(
    "10",
    check=[
        "Which credential did each answer name for Q1? Find the 23:36:15 row in "
        "case/badge-log.csv and check.",
        "Open case/site-facts.md and read the hours each override code is valid. Did each answer "
        "use them when it wrote COULD or COULD NOT?",
        "In the 'Same puzzle, three runs' table, which run spent its whole budget thinking "
        "and left no answer?",
    ],
    exercises=[
        "In run.py, change max_tokens=4000 in step 3's run(...) call to 8000, then run make "
        "lesson N=10 ARGS=--large. How long does it take, and is the answer right?",
        "In run.py, change max_tokens=300 in step 2's run(...) call to 1200. Does an answer "
        "appear, and is it any better than thinking off?",
        "In run.py, inside run(), change chat(puzzle, ...) to chat(puzzle if think else puzzle + "
        "\" Think step by step.\", ...). Compare step 1's tokens and accuracy with step 3.",
        "In run.py, in `puzzle`, change Q2 to ask whether each candidate CAN or CANNOT open it, "
        "and put candidates e) and f) on one line. What changes?",
        "In run.py, delete the <other_facts> block from `puzzle`, tags and all. Which candidates "
        "can no longer be ruled out, by you or the model?",
    ],
)

finding("""
The door was not opened with a badge. It was opened with OVR-AUD-02, a contractor override code
that facilities issued to Brightwater & Co on 10 February for the audit and never revoked. The
lift contractor's code stopped working at 18:00. Petrov is on the hut camera from 23:29 to 23:44.
Voss was at a dinner with several hundred witnesses. Reyes was already inside the stairwell.

Brightwater's audit was led by Simon Clarke. He is the man who signed off Halcyon's accounts, and
the man who texted Reyes at 21:44 to ask if he was still at the office.
""")
save_shot("10-thinking-mode")

## The nuances

**1. Thinking spends your output budget first.** At 300 tokens the model had only just worked out
Q1 when the budget ran out, mid-sentence: 963 characters of thinking and an empty answer. With thinking on,
always raise `max_tokens`, and treat an empty `reply.text` as "budget exhausted", not "no answer".

**2. Thinking is slow on a laptop.** Thinking off: 700 tokens in 8 seconds. Thinking on: 2,184
tokens in 26 seconds. The seconds are wall clock per call (model load, prompt processing and
generation) on an Apple M1 Pro 16 GB. Most of the thinking run's output is reasoning nobody reads.
On a laptop, thinking mode is a choice you make per task, not a default.

**3. Thinking helps, and still does not make the answer right.** With 4,000 tokens to think, the
1.7B model sorted the two override codes correctly, named Brightwater & Co as the holder of
OVR-AUD-02, ruled out the other four people and the lift contractor, and said Brightwater must be
interviewed first. Thinking off, it had written COULD for a) to d) and invented that Malhotra and
Voss were inside the stairwell. But the thinking answer ruled out Malhotra with "No override code
held by R. Malhotra", and the records cannot rule him out: nothing places him elsewhere, and he
could have had the code. An earlier run with the same budget did worse, deciding that OVR-AUD-02
was "valid from 10 February to 18:00" and that nobody could have opened the door. Thinking changes
the odds; a check in code is still what tells you which run you got.

**4. A bigger model thinks longer, and the length is not predictable.** On `qwen3:4b`
(`make lesson N=10 ARGS=--large`, not repeated in the current run; an earlier run on a 28-core CPU,
timed as generation only) the final version of this puzzle used the whole 4,000 token budget on
thinking and returned nothing, after 432 seconds. On an earlier run with almost the same
wording, the same model finished in 3,905 tokens (405 seconds) and got the credential and four of
the six candidates right, then still named Petrov for interview. Thinking length varies run to run
and prompt to prompt. Budget for the long tail, not the average.

**5. Read the thinking, but do not cite it.** `reply.thinking` is useful for debugging: you can see
exactly where the 1.7B model worked out that OVR-MNT-01 had expired at 18:00 and OVR-AUD-02 was
never revoked. It is not evidence and not always a
faithful account of how the answer was produced. Hale will ask about the answer, not the musings.

**6. Ambiguous questions defeat thinking too.** The first version of this puzzle asked who "CAN"
open the door, and listed "the lift maintenance contractor, Brightwater & Co" on one line. Models
read that as one party, and answered "can" by badge level rather than by who was where. Rewriting
it as "could they have been the one who opened it" with a numbered list of six candidates helped
more than any setting.

**7. Is thinking mode still relevant?** Yes, and increasingly so. Most frontier models are now
hybrid or reasoning-first, and the questions have moved from "should I prompt for reasoning" to
"how much reasoning budget does this task deserve". Turn it on for multi-step logic, planning,
maths and puzzles like this one. Turn it off for extraction, classification, formatting and
anything latency sensitive, where it adds cost and little accuracy.

**8. Set a temperature for thinking.** Qwen recommends temperature 0.6 with thinking and warns
against greedy decoding (temperature 0), which can make reasoning loop. Other vendors fix the
sampling settings for you when thinking is on.

## On a hosted model

```bash
make lesson N=10 TIER=hosted     # needs ANTHROPIC_API_KEY in .env and `uv sync --extra cloud`
make lesson N=10 TIER=hosted REPLAY=1   # once a run has been recorded, no key needed
```

On the hosted path `think=True` becomes Claude's adaptive thinking (`display: summarized`) and
`think=False` becomes thinking disabled on claude-haiku-5-5, as `thinking_off` in `config.yaml`
sets. `--large` resolves to claude-sonnet-5-5. Thinking tokens still come out of the same
`max_tokens`, so the 300-token step tests the same budget problem. The thinking text Claude
returns is a summary, so the thinking characters are not comparable with Qwen's full trace. Claude
does not take `temperature`, `top_k`, `top_p` or `seed`: the 0.6 and 0 in `run()` are left out and
reported by `cloud.py`, so token counts will not repeat run to run as they do locally. No step is
skipped.

| Measure | Local: qwen3:1.7b, Ollama, Apple M1 Pro 16 GB | Hosted: claude-haiku-5-5 |
|---|---|---|
| Think off, 700: answer chars, tokens out | 2,260, 700 | not run |
| Think on, 300: thinking chars, answer chars | 963, 0 | not run |
| Think on, 4000: thinking chars, tokens out | 6,308, 2,184 | not run |
| Think on, 4000: answer chars, wall-clock seconds | 1,534, 26 | not run |

What the hosted run will test: whether thinking at length and still ruling out Malhotra, whom the
records cannot rule out, is a small-model effect. That thinking tokens are spent from the same budget
as the answer, so a small budget can return an empty answer, should hold on both.

## Try this

1. Run `--large` with step 3's budget raised to 8000. How long does it take, and is the answer
   right?
2. Set step 2's budget to 1200. Does an answer appear, and is it any better than thinking off?
3. Ask step 1 (`think=False`) to "think step by step" (chapter 09). Compare tokens and accuracy with
   step 3.
4. Put back the ambiguous wording: "CAN or CANNOT open it" with candidate e) and f) on one line.
   What changes?
5. Remove the `<other_facts>` block. Which candidates can no longer be ruled out, by you or the
   model?

## Master it

After this chapter you can say where thinking tokens come from and what they do to a budget,
predict an empty answer before it happens, decide per task whether thinking is worth its time, and
write the deterministic check that scores a thinking model's answer.

**Predict, then see.** `make lesson N=10` stops three times to ask what will happen: what a
300-token thinking run returns, how many tokens the 4,000-token run spends, and how many of the
six candidates the best LANTERN answer gets right against the check in step 5. Each answer is
checked against your own run. Add `ARGS=--no-predict` to watch without stopping.

**Build it.** Write the door check in [`practice/exercise.py`](practice/exercise.py): from the
badge log, the override codes and who was where, return who could have opened the door, who is
ruled out and why, and who cannot be ruled out.

```bash
make exercise N=10          # 8 offline tests on the real badge log, no model needed
make hint N=10              # one hint at a time
make solution N=10          # the reference, once you have tried
```

**Check yourself.** `make quiz N=10`: seven questions, each with the reason behind the answer.

**Break it.** `make challenge N=10` checks your latest run against three targets: a thinking run
with just enough budget to answer (1,500 tokens or fewer); a LANTERN answer that agrees with the
check on five of the six candidates; and step 3 starved until it thinks and never answers.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 10

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 10

In [ ]:
%run ../../scripts/learn.py hint 10

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 10

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

The door was not opened with a badge. It was opened with OVR-AUD-02, a contractor override code
that facilities issued to Brightwater & Co on 10 February for the audit and never revoked. The lift
contractor's code stopped working at 18:00. Petrov is on the hut camera from 23:29 to 23:44. Voss
was at a dinner with several hundred witnesses. Reyes was already inside the stairwell.

Brightwater's audit was led by Simon Clarke. He is the man who signed off Halcyon's accounts, and
the man who texted Reyes at 21:44 to ask if he was still at the office.

**Next:** [Chapter 11: Rehearsal](../11-personas/README.md)